# UI Portal Data Model — Phase 1 Plan (MongoDB & GridFS)

**Scope of Phase 1:** the UI Portal (React) as the only source. Users log in, upload single or multiple files of the same schema, and the framework stages to MongoDB GridFS, validates against metadata contracts, and routes to Intake/Quarantine volumes. MongoDB is the system of record for ALL metadata; GridFS is the physical file store.

This plan is the **Phase-1 cut of the v1.3 consolidated design** (see `File Validation & Routing Framework — Design & Specs.ipynb`) — it does not replace it; it sequences and details it for portal-first delivery.

## Phase-1 principles

* **Same contracts, earlier subset** — the Phase-1 collections are identical in shape to the v1.3 catalog; nothing gets redesigned when later sources arrive. The `event_log` envelope is used from day one so SharePoint/SFTP/Volume adapters slot in without model changes.
* **MongoDB = metadata + files** — `file_registry` is the control entity; GridFS holds bytes only.
* **Everything idempotent** — checksum dedup, unique `(event_type, dedup_key)`, partial-unique transfer guard from the first deploy.
* **Audit everything** — config changes, status changes, and routing decisions are audited from day one.

## Out of scope for Phase 1 (designed but deferred)

* SharePoint / SFTP / Databricks-Volume detection adapters (Phase 2)
* Databricks pushback + reconciliation + archival purge gates (Phase 3 — GridFS copies are simply retained in Phase 1)
* CONTENT-category rules such as regional residency (Phase 2)

> **Status: DRAFT FOR REVIEW** — see the final cell for open questions that need your sign-off before build starts.

## Collection Inventory — What Phase 1 Builds

14 collections now, 7 deferred (shapes already fixed in the v1.3 design — no rework later).

| # | Collection | Layer | Phase | Role in the portal |
| --- | --- | --- | --- | --- |
| 1 | `users` | Registry | **1** | Portal identity + RBAC (UPLOADER / ADMIN / OPS roles) — new for portal |
| 2 | `source_systems` | Registry | **1** | Seeded with one doc: `src_portal` |
| 3 | `file_types` | Registry | **1** | Upload-screen dropdown; allowed extensions, size limits |
| 4 | `schema_versions` | Registry | **1** | Header fingerprint + column-order contract per file type |
| 5 | `validation_rules` | Registry | **1** | Rule registry (8 seed rules below) |
| 6 | `routing_policies` | Registry | **1** | Default policies: ERROR→Quarantine, PASSED→Intake |
| 7 | `volume_map` | Registry | **1** | Intake/Quarantine paths per file type |
| 8 | `event_log` | Eventing | **1** | Portal emits canonical `FILE_ARRIVED` from day one (envelope reused by Phase-2 adapters) |
| 9 | `upload_sessions` | Execution | **1** | Who uploaded what, from where; multi-file completeness; resumable parts |
| 10 | `file_registry` | Execution | **1** | Central control entity + state machine |
| 11 | `validation_results` | Execution | **1** | Per file × rule outcomes + evidence |
| 12 | `file_transfers` | Execution | **1** | GridFS → volume manifest with checksum verify + idempotency |
| 13 | `audit_log` | Governance | **1** | Append-only config/status/routing trail |
| 14 | `file_lifecycle` | Governance | **1** | Physical location history (GRIDFS → INTAKE_VOLUME / QUARANTINE) |
| 15 | `notifications` | Governance | 1 (lite) | Email/Teams on transfer failure — wire-up optional at launch |
| 16 | `batches` | Execution | 2 | Portal is synchronous; batches matter when scheduled/polling sources arrive |
| 17 | `retention_policies` / 18 `archive_jobs` | Governance | 3 | Archival + purge — gated by reconciliation |
| 19 | `databricks_ingestion_results` | Resilience | 3 | Databricks pushback |
| 20 | `reconciliation_runs` / 21 `reconciliation_results` | Resilience | 3 | Resilience certification; also the GridFS purge gate |

## Portal-Specific Collections (new detail beyond v1.3)

### `users` — portal identity + RBAC

```json
{
  "user_id": "usr_01HV4A...",
  "email": "analyst@corp.com",
  "display_name": "A. Analyst",
  "roles": ["UPLOADER", "ADMIN"],
  "department": "finance",
  "is_active": true,
  "created_at": {"$date": "2026-11-09T09:00:00Z"},
  "last_login_at": {"$date": "2026-11-09T09:12:00Z"}
}
```

* **Auth itself** is expected to be SSO/OIDC (e.g. Entra ID) — the portal validates tokens; `users` caches the profile + framework roles. Index: unique on `email`.
* Roles gate the React routes: `UPLOADER` (upload screens), `OPS` (status board, reclassify), `ADMIN` (rules/schemas/routing CRUD + approvals).
* **Open question (see final cell):** roles managed here, or mapped from IdP groups?

### `upload_sessions` — one portal interaction, many files

```json
{
  "session_id": "ses_01HV4B...",
  "source_id": "src_portal",
  "user_id": "usr_01HV4A...",
  "user_email": "analyst@corp.com",
  "client_ip": "203.0.113.10",
  "machine_name": "LT-4412",
  "user_agent": "Mozilla/5.0 (Windows NT 10.0) ...",
  "auth_method": "SSO_OIDC",
  "roles_snapshot": ["UPLOADER"],
  "files_declared_count": 3,
  "files_received_count": 3,
  "parts": [
    { "file_id": "file_01HV4C...", "gridfs_file_id": "665f1a2e...", "status": "COMPLETED", "chunks_received": 41 },
    { "file_id": "file_01HV4D...", "gridfs_file_id": "665f1b31...", "status": "COMPLETED", "chunks_received": 7 },
    { "file_id": "file_01HV4E...", "gridfs_file_id": "665f1c99...", "status": "COMPLETED", "chunks_received": 12 }
  ],
  "started_at": {"$date": "2026-11-09T09:14:00Z"},
  "completed_at": {"$date": "2026-11-09T09:14:52Z"}
}
```

* **Upload-completeness check for free:** `files_declared_count` (the React client declares the manifest before the first byte) vs `files_received_count` — mismatch = visible warning to the user + flagged in the ops view.
* **Resumable uploads:** `parts[]` tracks per-file chunk progress against GridFS; a dropped connection resumes where `chunks_received` stopped.
* `roles_snapshot` freezes the user's roles at upload time — audit-safe even if roles change later.

## Core Document Designs (Phase-1 shapes)

### `file_registry` — populated by a portal upload

```json
{
  "file_id": "file_01HV4C...",
  "source_id": "src_portal",
  "original_name": "customer_master_20261109.csv",
  "staged_path": null,
  "gridfs_file_id": "665f1a2e...",
  "size_bytes": 10485760,
  "extension": ".csv",
  "mime_type": "text/csv",
  "sha256_checksum": "9f86d081884c7d659a2feaa0d554f64e...",
  "file_type_id": "ft_customer",
  "schema_version_id": "sch_customer_v3",
  "identification_method": "EXPLICIT",
  "identification_confidence": 1.0,
  "identification_evidence": { "ui_selection": "Customer Master", "header_match": 22 },
  "unidentified_reason": null,
  "upload_session_id": "ses_01HV4B...",
  "uploaded_by": "usr_01HV4A...",
  "client_ip": "203.0.113.10",
  "machine_name": "LT-4412",
  "current_status": "ROUTED_INTAKE",
  "batch_id": null,
  "retry_count": 0,
  "received_at": {"$date": "2026-11-09T09:14:10Z"},
  "updated_at": {"$date": "2026-11-09T09:14:48Z"}
}
```

Portal uploads are `identification_method: EXPLICIT` (the user picks the file type), so Phase 1 never needs filename regex or content sniffing — those activate in Phase 2 for the unannounced sources.

### GridFS layout

* **One default bucket (`fs`)** — `fs.files` / `fs.chunks`, chunk size 255 KB (raise to 1 MB if typical files exceed ~100 MB).
* `fs.files` metadata is **minimal, reverse-lookup only**:

```json
{
  "metadata": {
    "file_id": "file_01HV4C...",
    "sha256": "9f86d081...",
    "original_name": "customer_master_20261109.csv",
    "upload_session_id": "ses_01HV4B..."
  }
}
```

* **No framework state in GridFS** — `file_registry` is the single source of truth; GridFS metadata exists only to find the registry doc from a chunk-level investigation.
* Phase 1 retention: GridFS copies are kept indefinitely (purge gates arrive with reconciliation in Phase 3).

### `validation_rules` — the 8 Phase-1 seed rules

| rule_code | category | severity | params (example) |
| --- | --- | --- | --- |
| `FILE_TYPE_ALLOWED` | FILE | ERROR | `{"allowed_extensions": [".csv", ".xlsx"]}` (also inherited from `file_types`) |
| `FILE_SIZE_MAX` | FILE | ERROR | `{"max_mb": 500}` |
| `FILE_NOT_EMPTY` | FILE | ERROR | `{"min_bytes": 1}` |
| `FILE_NOT_DUPLICATE` | FILE | ERROR | `{"scope": "global"}` (sha256 unique index) |
| `HEADER_REQUIRED` | HEADER | ERROR | `{"header_row": 1}` |
| `SCHEMA_MATCH` | HEADER | ERROR | `{"match_mode": "names", "ignore_case": true}` |
| `COLUMN_ORDER_MATCH` | HEADER | ERROR | `{"strict": true}` |
| `FILE_NOT_EMPTY_ROWS` | HEADER | WARN | `{"min_rows": 1}` (shadow-mode example — launch as WARN, flip later) |

Each binds to `file_types` via `applies_to` (or `null` = global). New rules = insert a doc; new rule kinds = one `BaseValidator` subclass (see master design).

### `routing_policies` — Phase-1 defaults (first match by priority)

| priority | match | action |
| --- | --- | --- |
| 10 | any rule `FAIL` with severity ERROR | `MOVE_QUARANTINE` (reason bundle attached) |
| 20 | `current_status: UNIDENTIFIED` | `MOVE_QUARANTINE` (unidentified subfolder) |
| 30 | `current_status: PASSED` or `WARNED` | `MOVE_INTAKE` |

### `volume_map` — seeds

| file_type | destination | path |
| --- | --- | --- |
| customer | INTAKE | `/Volumes/<cat>/<schema>/intake/customer/` |
| order | INTAKE | `/Volumes/<cat>/<schema>/intake/order/` |
| billing | INTAKE | `/Volumes/<cat>/<schema>/intake/billing/` |
| product | INTAKE | `/Volumes/<cat>/<schema>/intake/product/` |
| employee | INTAKE | `/Volumes/<cat>/<schema>/intake/employee/` |
| * (all types) | QUARANTINE | `/Volumes/<cat>/<schema>/quarantine/` |

Naming at transfer: `{file_type}_{file_id}_{ts}.ext` — bronze will read `file_id` straight from `_metadata.file_name` in Phase 3.

## State Machine & Event Envelope (portal subset)

```text
RECEIVED → IDENTIFYING (instant for portal: EXPLICIT) → VALIDATING
  → PASSED  → ROUTED_INTAKE
  → WARNED  → ROUTED_INTAKE (flagged)
  → FAILED  → ROUTED_QUARANTINE
RECEIVED → UNIDENTIFIED → (manual re-classification) → VALIDATING
any state → FRAMEWORK_ERROR → retry → previous state
```

Illegal transitions rejected by the API layer and logged to `audit_log`.

**Why keep `event_log` in Phase 1 although the portal is synchronous?** The upload API emits the canonical `FILE_ARRIVED` event and the pipeline worker consumes it — the same envelope the SharePoint/SFTP/Volume adapters will publish in Phase 2. This keeps the core processing loop source-agnostic from the first deploy.

Portal `FILE_ARRIVED` dedup key: `session_id|filename|part_index`.

## API ↔ Collection Mapping (Phase-1 contract)

| Endpoint | Method | Writes | Reads |
| --- | --- | --- | --- |
| `/api/auth/profile` | GET | — | `users` |
| `/api/file-types` | GET | — | `file_types`, `schema_versions` |
| `/api/uploads/manifest` | POST | `upload_sessions` (declared counts) | — |
| `/api/uploads/{session_id}/parts` | POST (chunk) | GridFS, `upload_sessions.parts[]` | — |
| `/api/uploads/{session_id}/complete` | POST | `file_registry`, `event_log` (FILE_ARRIVED) | — |
| `/api/files/{file_id}` | GET | — | `file_registry` + `validation_results` (results screen) |
| `/api/files/{file_id}/reclassify` | POST | `file_registry`, `audit_log` | `file_types` |
| `/api/ops/files` | GET | — | `file_registry` (status board, filters) |
| `/api/ops/sessions` | GET | — | `upload_sessions` |
| `/api/ops/events/dead` | GET / POST re-drive | `event_log` | `event_log` |
| `/api/admin/rules` `/schemas` `/routing` `/volumes` | CRUD | respective registry collections + `audit_log` (with `approver`) | — |

### React pages ↔ collections

| Portal screen | Driven by |
| --- | --- |
| **Upload** (type picker, multi-file, resumable progress) | `file_types`, `upload_sessions.parts[]` |
| **My uploads / results** | `upload_sessions`, `file_registry`, `validation_results` |
| **Ops status board** (quarantined, unidentified, failed transfers) | `file_registry`, `file_transfers`, `event_log` (DEAD) |
| **Admin config** (rules, schemas, routing, volumes) | `validation_rules`, `schema_versions`, `routing_policies`, `volume_map`, `audit_log` |

Note: FastAPI stays the **only** writer to MongoDB — the React client never touches Mongo directly.

## Phase-1 Index Strategy

Reuse the master `$jsonSchema` validators and indexes from the design doc (`MongoDB Index & Schema Validation Specs` cell). Phase-1 essentials, in build order:

| Collection | Index | Purpose |
| --- | --- | --- |
| `users` | `{email: 1}` unique | identity |
| `file_registry` | `{sha256_checksum: 1}` unique | duplicate detection |
| `file_registry` | `{source_id: 1, received_at: -1}` | ops board |
| `file_registry` | `{current_status: 1}` | status filters |
| `file_registry` | `{upload_session_id: 1}` | my-uploads screen |
| `validation_results` | `{file_id: 1, executed_at: -1}` | results screen |
| `event_log` | `{event_type: 1, dedup_key: 1}` unique | idempotent processing |
| `file_transfers` | `{file_id: 1, direction: 1}` unique, partial on `status: COMPLETED` | never transfer twice |
| `upload_sessions` | `{user_id: 1, started_at: -1}` | my-uploads history |
| `audit_log` | `{entity_id: 1, ts: -1}` | audit drill-down |

Validators: `validationLevel: strict` / `validationAction: error` on all write-critical collections — full specs already exist in the master design; no re-authoring needed.

## Build Sequence & Review Checkpoints

| Step | Deliverable | Review checkpoint |
| --- | --- | --- |
| 1 | Seed registry collections: `source_systems` (src_portal), `users`, `file_types`, `schema_versions`, 8 `validation_rules`, 3 `routing_policies`, `volume_map` | **R1 — schema review** (you sign off on contracts & seeds before any code) |
| 2 | Create operational collections with `$jsonSchema` validators + indexes | validators run against Step-1 seeds |
| 3 | FastAPI upload API: manifest → chunked parts → complete → `file_registry` + GridFS + `event_log` | resumable upload happy path |
| 4 | Rule engine: `BaseValidator` + registry + the 8 seed rules (FILE → HEADER) | **R2 — dry run on sample files** (good, bad, dup, wrong-order headers) |
| 5 | Routing + transfer worker: policy evaluation → `file_transfers` → volume write, checksum verify | quarantine + intake paths in the Databricks volume |
| 6 | Ops + Admin APIs | API contract freeze |
| 7 | React pages: upload, results, ops board, admin | **R3 — end-to-end walkthrough** |
| 8 | Optional: notifications wiring, WARN-rule shadow reports | launch checklist |

**Definition of done (Phase 1):** a multi-file portal upload is validated, quarantined-or-intaked per policy, fully audited (user/IP/machine + rule evidence + transfer checksums), and visible in the ops board — with zero schema rework needed for Phase-2 sources.

## Open Questions for Review

1. **Identity** — is the portal auth SSO/OIDC (e.g. Entra ID)? Then: are roles managed in `users`, or mapped from IdP groups?
2. **File-type mixing** — must a single upload session contain only ONE file type (current assumption: yes — same schema), or can a user mix customer + order files in one session?
3. **Size envelope** — max file size at launch (seed rule says 500 MB) and expected typical size (drives GridFS chunk size 255 KB vs 1 MB).
4. **Volume target** — confirm the intake volume root `/Volumes/<cat>/<schema>/intake` and its catalog/schema now, so `volume_map` seeds are final.
5. **Duplicate policy** — global checksum uniqueness (same file re-uploaded by ANY user is rejected), or per-user? (Seed assumes global.)
6. **WARN at launch** — should `FILE_NOT_EMPTY_ROWS` (or any other rule) ship in shadow mode initially?
7. **MongoDB deployment** — Atlas vs self-managed replica set (change streams require a replica set; affects the event worker).
8. **Quarantine UX** — who gets notified and who can re-classify unidentified files: any OPS user, or admin-only?

---
*Companion documents: `File Validation & Routing Framework — Design & Specs.ipynb` (v1.3 master design, 20-collection catalog, full validators/index specs) • repo `README.md`.*